## The Transformation Logic

In [0]:
from datetime import datetime

# ---------------------------------------------------------
# 1. Build today's suffix once
# ---------------------------------------------------------
today = datetime.today().strftime("%Y_%m_%d")

# ---------------------------------------------------------
# 2. Build ONLY the tables used in your query
# ---------------------------------------------------------
crm_customers = f"`abc_business-core`.silver.crm_customers_{today}"
erp_customers = f"`abc_business-core`.silver.erp_customers_{today}"
erp_locations = f"`abc_business-core`.silver.erp_locations_{today}"

# ---------------------------------------------------------
# 3. Your original query rewritten with dynamic table names
# ---------------------------------------------------------
query = f"""
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM {crm_customers} ci
LEFT JOIN {erp_customers} ca
    ON ci.customer_number = ca.customer_number
LEFT JOIN {erp_locations} la
    ON ci.customer_number = la.customer_number
"""

# ---------------------------------------------------------
# 4. Execute
# ---------------------------------------------------------
df = spark.sql(query)

In [0]:
df.limit(5).display()

### Writing Gold Table

In [0]:
table_name = f"`abc_business-core`.gold.dim_customers_{today}"
df.write.mode("overwrite").format("delta").saveAsTable(table_name)

### Sanity checks of Gold table

In [0]:
df = spark.sql(f"SELECT * FROM {table_name} LIMIT 5")
df.show()